# PAEwall — Dual-Encoder Training (Colab A100)

Trains a two-tower contrastive model for patent-to-product retrieval.

**Architecture:**
- Patent tower: `AI-Growth-Lab/PatentSBERTa` fine-tuned on (patent_claims, product_description) pairs
- Product tower: Shared or separate SentenceTransformer encoder
- Loss: InfoNCE (in-batch negatives + hard negatives from BM25)

**Runtime:** ~2–3 hours on A100 for text-only; ~4–5 hours for multimodal.

**Outputs saved to Google Drive:** `MyDrive/paewall/models/`

## 0. Setup

In [ ]:
# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
!pip install -q sentence-transformers rank-bm25 faiss-gpu loguru wandb

In [ ]:
# Mount Drive — PAE-Bench parquet and model outputs go here
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/paewall'
DATA_DIR  = f'{DRIVE_DIR}/data/processed'
MODEL_DIR = f'{DRIVE_DIR}/models/dual_encoder'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print('Drive mounted. Model outputs:', MODEL_DIR)

## 1. Load PAE-Bench

Upload `pae_bench.parquet` to `MyDrive/paewall/data/processed/` before running.

In [ ]:
import pandas as pd
import numpy as np

bench = pd.read_parquet(f'{DATA_DIR}/pae_bench.parquet')
print(f'PAE-Bench: {len(bench)} rows, {bench["patent_id"].nunique()} patents, {bench["company_name"].nunique()} companies')
print(f'Verticals: {bench["vertical"].value_counts().to_dict()}')
bench.head(3)

In [ ]:
from sklearn.model_selection import train_test_split

patent_ids = bench['patent_id'].unique()
train_ids, test_ids = train_test_split(patent_ids, test_size=0.2, random_state=42)
val_ids, test_ids  = train_test_split(test_ids,   test_size=0.5, random_state=42)

train_df = bench[bench['patent_id'].isin(train_ids)].reset_index(drop=True)
val_df   = bench[bench['patent_id'].isin(val_ids)].reset_index(drop=True)
test_df  = bench[bench['patent_id'].isin(test_ids)].reset_index(drop=True)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

## 2. Hard Negative Mining (BM25)

In-batch negatives alone are weak — BM25 mines *hard* negatives: products that look similar to the patent text but are NOT the true infringer.

In [ ]:
import re
from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list:
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return text.split()

product_corpus = (
    bench[['company_name', 'product_description']]
    .drop_duplicates('company_name')
    .reset_index(drop=True)
)

print(f'Building BM25 index over {len(product_corpus)} products...')
bm25 = BM25Okapi([tokenize(t) for t in product_corpus['product_description'].fillna('')])
print('Done.')

def mine_hard_negatives(patent_claims: str, positive_company: str, k: int = 5) -> list[str]:
    """Return top-k BM25 products that are NOT the positive match."""
    scores = bm25.get_scores(tokenize(patent_claims))
    top_idx = np.argsort(scores)[::-1][:k + 10]
    results = []
    for i in top_idx:
        name = product_corpus['company_name'].iloc[i]
        if name.lower() != positive_company.lower():
            results.append(product_corpus['product_description'].iloc[i])
        if len(results) >= k:
            break
    return results

## 3. Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader

class PatentProductDataset(Dataset):
    """
    Each item is a (patent_claims, positive_product, [hard_neg_1, ..., hard_neg_k]) tuple.
    Returns text — tokenization happens inside the collate function.
    """
    def __init__(self, df: pd.DataFrame, hard_neg_k: int = 3):
        # One row per unique patent (group by patent_id)
        self.records = []
        for patent_id, group in df.groupby('patent_id'):
            claims = str(group['patent_claims'].iloc[0] or '')
            pos_company = group['company_name'].iloc[0]
            pos_desc = str(group['product_description'].iloc[0] or '')
            hard_negs = mine_hard_negatives(claims, pos_company, k=hard_neg_k)
            self.records.append({
                'patent_claims': claims[:2048],   # truncate to avoid OOM
                'positive': pos_desc[:512],
                'hard_negatives': [h[:512] for h in hard_negs],
            })

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        return self.records[idx]


print('Building train dataset (mining hard negatives)...')
train_dataset = PatentProductDataset(train_df, hard_neg_k=3)
val_dataset   = PatentProductDataset(val_df,   hard_neg_k=3)
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')

## 4. Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PATENT_MODEL  = 'AI-Growth-Lab/PatentSBERTa'   # domain-tuned on patents
PRODUCT_MODEL = 'sentence-transformers/all-mpnet-base-v2'  # general purpose
EMBED_DIM = 768


def mean_pool(model_output, attention_mask):
    """Mean-pool token embeddings weighted by attention mask."""
    token_emb = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_emb.size()).float()
    return (token_emb * mask).sum(1) / mask.sum(1).clamp(min=1e-9)


class DualEncoder(nn.Module):
    """
    Two-tower encoder for patent-to-product retrieval.
    Patent tower: PatentSBERTa (domain-tuned)
    Product tower: all-mpnet-base-v2 (general)
    Both project into a shared 256-dim embedding space.
    """
    def __init__(self, patent_model_name: str, product_model_name: str,
                 proj_dim: int = 256, temperature: float = 0.07):
        super().__init__()
        self.patent_encoder  = AutoModel.from_pretrained(patent_model_name)
        self.product_encoder = AutoModel.from_pretrained(product_model_name)
        self.patent_proj  = nn.Linear(EMBED_DIM, proj_dim)
        self.product_proj = nn.Linear(EMBED_DIM, proj_dim)
        self.temperature = nn.Parameter(torch.tensor(temperature))

    def encode_patents(self, input_ids, attention_mask):
        out = self.patent_encoder(input_ids=input_ids, attention_mask=attention_mask)
        emb = mean_pool(out, attention_mask)
        return F.normalize(self.patent_proj(emb), dim=-1)

    def encode_products(self, input_ids, attention_mask):
        out = self.product_encoder(input_ids=input_ids, attention_mask=attention_mask)
        emb = mean_pool(out, attention_mask)
        return F.normalize(self.product_proj(emb), dim=-1)

    def forward(self, patent_inputs, positive_inputs, negative_inputs_list=None):
        """
        Returns InfoNCE loss.
        patent_inputs, positive_inputs: dicts with input_ids + attention_mask
        negative_inputs_list: list of hard-negative input dicts
        """
        p_emb  = self.encode_patents(**patent_inputs)    # (B, D)
        pos_emb = self.encode_products(**positive_inputs) # (B, D)

        # Build key matrix: positives + hard negatives
        keys = [pos_emb]  # (B, D)
        if negative_inputs_list:
            for neg_inputs in negative_inputs_list:
                neg_emb = self.encode_products(**neg_inputs)
                keys.append(neg_emb)

        # keys: (B, 1+K, D) -> (B*(1+K), D)
        keys = torch.stack(keys, dim=1)  # (B, 1+K, D)
        B, NK, D = keys.shape

        # Similarity: (B, B*(1+K))
        keys_flat = keys.view(B * NK, D)  # (B*(1+K), D)
        logits = (p_emb @ keys_flat.T) / self.temperature.exp()  # (B, B*(1+K))

        # Labels: diagonal positions (index 0, NK+1, 2*NK+2, ...)
        labels = torch.arange(B, device=logits.device) * NK
        loss = F.cross_entropy(logits, labels)
        return loss


print(f'Loading models on {DEVICE}...')
model = DualEncoder(PATENT_MODEL, PRODUCT_MODEL).to(DEVICE)
patent_tokenizer  = AutoTokenizer.from_pretrained(PATENT_MODEL)
product_tokenizer = AutoTokenizer.from_pretrained(PRODUCT_MODEL)
print('Models loaded.')

## 5. Training

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm

# --- Hyperparameters ---
BATCH_SIZE   = 16
EPOCHS       = 5
LR           = 2e-5
MAX_SEQ_PAT  = 512   # patent claims (long)
MAX_SEQ_PROD = 256   # product description
HARD_NEG_K   = 3


def tokenize_batch(texts, tokenizer, max_length):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    ).to(DEVICE)


def collate_fn(batch):
    patent_texts   = [item['patent_claims'] for item in batch]
    positive_texts = [item['positive']      for item in batch]
    hard_neg_texts = [[item['hard_negatives'][k] if k < len(item['hard_negatives']) else ''
                       for item in batch]
                      for k in range(HARD_NEG_K)]
    return {
        'patent_inputs':  tokenize_batch(patent_texts,   patent_tokenizer,  MAX_SEQ_PAT),
        'positive_inputs': tokenize_batch(positive_texts, product_tokenizer, MAX_SEQ_PROD),
        'negative_inputs_list': [
            tokenize_batch(neg_texts, product_tokenizer, MAX_SEQ_PROD)
            for neg_texts in hard_neg_texts
        ],
    }


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))

print(f'Training: {len(train_loader)} batches/epoch x {EPOCHS} epochs')

In [ ]:
best_val_loss = float('inf')
history = []

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    train_losses = []
    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [train]'):
        optimizer.zero_grad()
        loss = model(
            patent_inputs=batch['patent_inputs'],
            positive_inputs=batch['positive_inputs'],
            negative_inputs_list=batch['negative_inputs_list'],
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_losses.append(loss.item())

    # --- Validate ---
    model.eval()
    val_losses = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch}/{EPOCHS} [val]'):
            loss = model(
                patent_inputs=batch['patent_inputs'],
                positive_inputs=batch['positive_inputs'],
                negative_inputs_list=batch['negative_inputs_list'],
            )
            val_losses.append(loss.item())

    train_loss = np.mean(train_losses)
    val_loss   = np.mean(val_losses)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
    print(f'Epoch {epoch}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}')

    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint = {
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_loss': val_loss,
            'config': {
                'patent_model': PATENT_MODEL,
                'product_model': PRODUCT_MODEL,
                'batch_size': BATCH_SIZE,
                'lr': LR,
                'hard_neg_k': HARD_NEG_K,
            }
        }
        torch.save(checkpoint, f'{MODEL_DIR}/best_model.pt')
        print(f'  -> Saved best model (val_loss={val_loss:.4f})')

## 6. Evaluation on PAE-Bench (Recall@K, MRR, nDCG@10)

In [ ]:
# Load best checkpoint
ckpt = torch.load(f'{MODEL_DIR}/best_model.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded best model from epoch {ckpt["epoch"]} (val_loss={ckpt["val_loss"]:.4f})')

In [ ]:
import faiss

@torch.no_grad()
def encode_all_products(product_corpus: pd.DataFrame, batch_size: int = 64) -> np.ndarray:
    """Encode all products into the embedding space."""
    all_embs = []
    texts = product_corpus['product_description'].fillna('').tolist()
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenize_batch(batch_texts, product_tokenizer, MAX_SEQ_PROD)
        embs = model.encode_products(**inputs).cpu().numpy()
        all_embs.append(embs)
    return np.vstack(all_embs)


@torch.no_grad()
def encode_patent(claims_text: str) -> np.ndarray:
    inputs = tokenize_batch([claims_text], patent_tokenizer, MAX_SEQ_PAT)
    return model.encode_patents(**inputs).cpu().numpy()


def build_faiss_index(product_embs: np.ndarray) -> faiss.Index:
    dim = product_embs.shape[1]
    index = faiss.IndexFlatIP(dim)  # inner product on normalized vecs = cosine sim
    faiss.normalize_L2(product_embs)
    index.add(product_embs)
    return index


def evaluate_retrieval(benchmark: pd.DataFrame, product_corpus: pd.DataFrame,
                       index: faiss.Index, top_k: int = 50) -> dict:
    recall_10, recall_50, mrr_scores, ndcg_10 = [], [], [], []

    for patent_id, group in benchmark.groupby('patent_id'):
        claims = str(group['patent_claims'].iloc[0] or '')
        relevant = set(group['company_name'].str.lower())

        q_emb = encode_patent(claims)
        faiss.normalize_L2(q_emb)
        _, indices = index.search(q_emb, top_k)
        retrieved = [product_corpus['company_name'].iloc[i].lower() for i in indices[0]]

        r10 = len(set(retrieved[:10]) & relevant) / max(len(relevant), 1)
        r50 = len(set(retrieved[:50]) & relevant) / max(len(relevant), 1)
        recall_10.append(r10)
        recall_50.append(r50)

        mrr = next((1.0/(rank+1) for rank, c in enumerate(retrieved) if c in relevant), 0.0)
        mrr_scores.append(mrr)

        dcg  = sum(1.0/np.log2(r+2) for r,c in enumerate(retrieved[:10]) if c in relevant)
        idcg = sum(1.0/np.log2(i+2) for i in range(min(len(relevant), 10)))
        ndcg_10.append(dcg / max(idcg, 1e-10))

    return {
        'Recall@10': float(np.mean(recall_10)),
        'Recall@50': float(np.mean(recall_50)),
        'MRR':       float(np.mean(mrr_scores)),
        'nDCG@10':   float(np.mean(ndcg_10)),
        'n_queries': len(recall_10),
    }


print('Encoding all products...')
product_embs = encode_all_products(product_corpus)
print(f'Product embeddings: {product_embs.shape}')

print('Building FAISS index...')
index = build_faiss_index(product_embs.copy())

print('Evaluating on test set...')
test_metrics = evaluate_retrieval(test_df, product_corpus, index)
print('\n=== Dual-Encoder (text-only) — Test Results ===')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

## 7. Ablation: Text-only vs Multimodal

This cell records the text-only numbers. Run the multimodal version (adding SigLIP vision towers) separately and compare.

In [ ]:
import json

# Per-vertical breakdown
per_vertical = {}
for vertical in test_df['vertical'].unique():
    subset = test_df[test_df['vertical'] == vertical]
    if len(subset) < 5:
        continue
    v_metrics = evaluate_retrieval(subset, product_corpus, index)
    per_vertical[vertical] = v_metrics
    print(f'  [{vertical}] Recall@10: {v_metrics["Recall@10"]:.4f}  MRR: {v_metrics["MRR"]:.4f}')

results = {
    'model': 'dual_encoder_text_only',
    'config': ckpt['config'],
    'overall': test_metrics,
    'per_vertical': per_vertical,
    'training_history': history,
}

out_path = f'{MODEL_DIR}/eval_results_text_only.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved results to {out_path}')

## 8. Save FAISS Index + Encoders for Inference

In [ ]:
import pickle

# Save FAISS index
faiss.write_index(index, f'{MODEL_DIR}/product_index.faiss')

# Save product corpus (needed to map index -> company name)
product_corpus.to_parquet(f'{MODEL_DIR}/product_corpus.parquet', index=False)

# Save product embeddings for analysis
np.save(f'{MODEL_DIR}/product_embeddings.npy', product_embs)

print('Saved:')
print(f'  {MODEL_DIR}/best_model.pt')
print(f'  {MODEL_DIR}/product_index.faiss')
print(f'  {MODEL_DIR}/product_corpus.parquet')
print(f'  {MODEL_DIR}/product_embeddings.npy')
print('\nDownload these to models/dual_encoder/ in the repo for inference.')